# 主流训练框架 API 对比

> 第 13 节算清了 ZeRO 三阶段把显存冗余逐层切到多卡的原理，也指出 DDP / FSDP / DeepSpeed / Accelerate 处在不同的抽象层级。但停留在原理层面还不够：真到了配置文件和训练循环那一行，每个框架暴露给用户的 API 形状并不一致。
>
> 这一节按抽象层级梳理主流训练框架的 API。从 PyTorch 原生的 DDP / FSDP / pipeline.sync 出发，看清它们的构造函数签名、关键参数和最小用法；再向上走到 DeepSpeed 的配置驱动模式、Accelerate 的统一封装；最后讨论 Megatron-LM 这套自建层的代码结构，以及 MoE 场景下各框架的成熟度差异。

训练框架的 API 长什么样，决定了我们写训练脚本时的思维负担。低层 API（DDP、FSDP）要求显式处理进程组、设备、梯度同步；高层 API（Accelerate、TRL）把这些藏进一个对象或一份配置，代价是对底层的控制力下降。本节的目标是给出一份按抽象层级排列的 API 地图，让读者拿到一个新项目时能快速判断「这个场景应该用哪一层」。

先约定一些术语。**并行策略（parallelism strategy）**指数据并行、张量并行、流水并行、专家并行这些切分维度的算法描述；**框架（framework）**指把这些策略封装成 API 的具体实现，比如 PyTorch FSDP、DeepSpeed、Megatron-LM。同一个策略往往有多套实现，实现之间的差异主要体现在配置方式、通信优化、对新硬件特性的支持速度上。

本节所有 demo 都用 gloo 后端在单机两进程下可跑，或标注为伪代码。重点是 API 形状，不是性能。

## 1. API 层次全景

训练分布式相关的 API 大致分为四层，从下往上抽象度提升、灵活性下降：

| 层级 | 代表 API | 形态 | 用户责任 |
|:---|:---|:---|:---|
| L0 primitive | `torch.distributed.init_process_group` | 进程组、collective op | 自己写 all-reduce、broadcast |
| L1 并行模块 | `DistributedDataParallel`、`FullyShardedDataParallel`、`pipeline.sync.Pipe` | nn.Module 包装类 | 自己组装训练循环 |
| L2 框架层 | DeepSpeed、Accelerate、TRL | engine 或 accelerator 对象 | 配置文件 + 简短循环 |
| L3 自建层 | Megatron-LM | 自定义层 + 训练脚本 | 阅读源码、定制修改 |

L0 是 D 主题（分布式 primitive），本节从 L1 开始向上走。L3 与前三层性质不同：Megatron-LM 不是一个被「import 进来调用」的库，而是一套需要 fork 或深度集成的训练代码基线。

## 2. DistributedDataParallel（DDP）

DDP 是 PyTorch 数据并行的标准实现。它的核心思想在第 13 节已经讲过：每张卡持有完整模型副本，反向结束后用 all-reduce 把梯度平均。这里看 API 形状。

构造函数签名（精简版）：

```python
torch.nn.parallel.DistributedDataParallel(
    module,
    device_ids=None,
    output_device=None,
    dim=0,
    broadcast_buffers=True,
    find_unused_parameters=False,
    gradient_as_bucket_view=False,
    static_graph=False,
)
```

三个最常调整的参数：

In [1]:
# === DDP 关键参数说明（不启动多进程，仅展示 API 形状） ===
import inspect
import torch.nn.parallel as par

params = {
    "device_ids": (
        "指定本进程使用的 GPU ID 列表。"
        "单卡训练传 [local_rank]；CPU 训练传 None。"
    ),
    "gradient_as_bucket_view": (
        "把梯度 bucket 暴露为视图。"
        "开启后 in-place 优化器能直接操作桶内存，"
        "减少一次拷贝，FP16 训练建议开启。"
    ),
    "find_unused_parameters": (
        "遍历 autograd 图找未参与 backward 的参数。"
        "MoE、条件计算场景必开，"
        "代价是每次 forward 都多一次图遍历。"
    ),
    "static_graph": (
        "假设每次 backward 的通信图拓扑不变。"
        "开启后能调度通信与计算重叠，"
        "固定结构的模型建议开启。"
    ),
}

for name, desc in params.items():
    print(f"{name}:")
    print(f"  {desc}")
    print()

device_ids:
  指定本进程使用的 GPU ID 列表。单卡训练传 [local_rank]；CPU 训练传 None。

gradient_as_bucket_view:
  把梯度 bucket 暴露为视图。开启后 in-place 优化器能直接操作桶内存，减少一次拷贝，FP16 训练建议开启。

find_unused_parameters:
  遍历 autograd 图找未参与 backward 的参数。MoE、条件计算场景必开，代价是每次 forward 都多一次图遍历。

static_graph:
  假设每次 backward 的通信图拓扑不变。开启后能调度通信与计算重叠，固定结构的模型建议开启。



下面是一个可在单机两进程下跑通的最小 DDP demo。用 `torch.multiprocessing` 启动两个进程，gloo 后端做 all-reduce。

In [2]:
# === DDP 最小 demo：gloo 后端，单机两进程 ===
# 这段代码完整可跑，用 spawn 启动两个 worker 进程
import os
import torch
import torch.multiprocessing as mp
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP


def worker(rank, world_size):
    """单个 DDP worker 的训练循环。"""
    # 用 gloo 后端，单机不需要 NCCL
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "29501"
    dist.init_process_group(
        backend="gloo",
        rank=rank,
        world_size=world_size,
    )

    # 一个 toy 模型：单层线性变换
    model = torch.nn.Linear(4, 2, bias=False)
    # DDP 包装：CPU 模式下 device_ids 传 None
    ddp_model = DDP(model, device_ids=None)

    # 不同 rank 喂不同输入，模拟数据并行
    torch.manual_seed(rank)
    x = torch.randn(2, 4)
    target = torch.zeros(2, 2)

    loss = ((ddp_model(x) - target) ** 2).mean()
    loss.backward()

    # DDP 自动 all-reduce 梯度，两个 rank 的 grad 应该相同
    grad_rank0 = ddp_model.module.weight.grad.clone()
    print(f"[rank {rank}] loss={loss.item():.4f}, grad[0,0]={grad_rank0[0,0].item():.4f}")

    dist.barrier()
    if rank == 0:
        print("关键观察：两个 rank 看到不同输入，但 backward 后梯度相同（all-reduce 已生效）。")
    dist.destroy_process_group()


if __name__ == "__main__":
    # macOS / Linux 单机直接 spawn
    mp.start_processes(worker, args=(2,), nprocs=2, join=True, start_method="fork")

[Gloo] Rank 1 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1
[Gloo] Rank 0 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1


[rank 0] loss=0.0436, grad[0,0]=-0.0957

[rank 1] loss=0.1529, grad[0,0]=-0.0957

关键观察：两个 rank 看到不同输入，但 backward 后梯度相同（all-reduce 已生效）。

DDP 的适用边界很明确：模型能完整放进单张卡时它是首选，通信开销最小、调试最简单。模型放不下时——比如 70B 全量训练——每张卡的固定开销就超过单卡显存，DDP 无法启动，必须切到 FSDP 或 ZeRO-3。

另一个不该用 DDP 的场景是 MoE。专家层只有部分参数在每次 forward 中被激活，DDP 默认假设所有参数都参与 backward，需要开启 `find_unused_parameters=True` 才能正确同步。但这个开关带来额外开销，工业界 MoE 训练一般直接上 FSDP + expert parallel。

## 3. FullyShardedDataParallel（FSDP）

FSDP 是 PyTorch 1.11 起内置的 ZeRO-3 实现。与 DDP 的关键区别是：DDP 每张卡持有完整模型副本，FSDP 把参数、梯度、优化器状态全部分片到各卡。前向和反向用到参数时，FSDP 自动 all-gather 出完整 layer，用完即丢。

构造函数签名（精简版）：

```python
torch.distributed.fsdp.FullyShardedDataParallel(
    module,
    sharding_strategy=None,
    cpu_offload=None,
    auto_wrap_policy=None,
    backward_prefetch=None,
    mixed_precision=None,
    sync_module_states=False,
    forward_prefetch=False,
    use_orig_params=True,
)
```

三个核心配置项：

In [3]:
# === FSDP 核心配置项说明 ===
from torch.distributed.fsdp import ShardingStrategy

strategies = {
    ShardingStrategy.FULL_SHARD: (
        "完整 ZeRO-3。参数/梯度/优化器全切片。"
        "显存最省，通信最重。"
    ),
    ShardingStrategy.SHARD_GRAD_OP: (
        "ZeRO-2 等价。切梯度和优化器，参数完整。"
        "通信比 FULL_SHARD 轻，显存比 DDP 省。"
    ),
    ShardingStrategy.NO_SHARD: (
        "退化为 DDP。不切分，只做梯度 all-reduce。"
        "调试或对比基线时用。"
    ),
    ShardingStrategy._HYBRID_SHARD_ZERO2: (
        "混合策略：机内 FULL_SHARD，机间 SHARD_GRAD_OP。"
        "适合多机集群，平衡机内/机间带宽差。"
    ),
}

print("ShardingStrategy 选项：")
for s, desc in strategies.items():
    print(f"  {s.name}:")
    print(f"    {desc}")
    print()

print("mixed_precision: 指定 param/grad/reduce 的 dtype 组合。")
print("  典型配置：param=BF16, grad=BF16, reduce=BF16（A100/H100 推荐）。")
print()
print("auto_wrap_policy: 自动决定哪些子模块用 FSDP 包装。")
print("  常用 size_based_auto_wrap_policy（按参数量）或 transformer_auto_wrap_policy（按层类型）。")

ShardingStrategy 选项：
  FULL_SHARD:
    完整 ZeRO-3。参数/梯度/优化器全切片。显存最省，通信最重。

  SHARD_GRAD_OP:
    ZeRO-2 等价。切梯度和优化器，参数完整。通信比 FULL_SHARD 轻，显存比 DDP 省。

  NO_SHARD:
    退化为 DDP。不切分，只做梯度 all-reduce。调试或对比基线时用。

  _HYBRID_SHARD_ZERO2:
    混合策略：机内 FULL_SHARD，机间 SHARD_GRAD_OP。适合多机集群，平衡机内/机间带宽差。

mixed_precision: 指定 param/grad/reduce 的 dtype 组合。
  典型配置：param=BF16, grad=BF16, reduce=BF16（A100/H100 推荐）。

auto_wrap_policy: 自动决定哪些子模块用 FSDP 包装。
  常用 size_based_auto_wrap_policy（按参数量）或 transformer_auto_wrap_policy（按层类型）。



# === FSDP 配置示例（结构展示，不实际启动多进程） ===
import torch
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    MixedPrecision,
    ShardingStrategy,
)
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy
import functools

# MixedPrecision 配置：A100/H100 推荐 BF16
mp_policy = MixedPrecision(
    param_dtype=torch.bfloat16,      # 参数在计算时 cast 成 BF16
    reduce_dtype=torch.bfloat16,     # 梯度 all-reduce 用 BF16
    buffer_dtype=torch.bfloat16,     # register_buffer 的张量也 cast
)

# auto_wrap_policy：参数量超过 1e6 的子模块单独包装
wrap_policy = functools.partial(
    size_based_auto_wrap_policy,
    min_num_params=1_000_000,
)

# 伪代码：真实训练中先 init_process_group 再调用
fsdp_init_code = '''
model = MyModel().cuda()
model = FSDP(
    model,
    sharding_strategy=ShardingStrategy.FULL_SHARD,
    mixed_precision=mp_policy,
    auto_wrap_policy=wrap_policy,
    use_orig_params=True,        # 保留原始参数名，方便 state_dict
    sync_module_states=True,     # rank 0 的权重同步到其他 rank
)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
'''

print("FSDP 初始化结构：")
print(fsdp_init_code)
print("关键观察：FSDP(model) 这一行就把 ZeRO-3 接好了，")
print("        训练循环和 DDP 几乎一致：loss.backward() + optimizer.step()。")


In [4]:
# === FSDP 配置示例（结构展示，不实际启动多进程） ===
import torch
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    MixedPrecision,
    ShardingStrategy,
)
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy
import functools

# MixedPrecision 配置：A100/H100 推荐 BF16
mp_policy = MixedPrecision(
    param_dtype=torch.bfloat16,      # 参数在计算时 cast 成 BF16
    reduce_dtype=torch.bfloat16,     # 梯度 all-reduce 用 BF16
    buffer_dtype=torch.bfloat16,     # register_buffer 的张量也 cast
)

# auto_wrap_policy：参数量超过 1e6 的子模块单独包装
wrap_policy = functools.partial(
    size_based_auto_wrap_policy,
    min_num_params=1_000_000,
)

# 伪代码：真实训练中先 init_process_group 再调用
fsdp_init_code = '''
model = MyModel().cuda()
model = FSDP(
    model,
    sharding_strategy=ShardingStrategy.FULL_SHARD,
    mixed_precision=mp_policy,
    auto_wrap_policy=wrap_policy,
    use_orig_params=True,        # 保留原始参数名，方便 state_dict
    sync_module_states=True,     # rank 0 的权重同步到其他 rank
)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
'''

print("FSDP 初始化结构：")
print(fsdp_init_code)
print("关键观察：FSDP(model) 这一行就把 ZeRO-3 接好了，")
print("        训练循环和 DDP 几乎一致：loss.backward() + optimizer.step()。")

FSDP 初始化结构：

model = MyModel().cuda()
model = FSDP(
    model,
    sharding_strategy=ShardingStrategy.FULL_SHARD,
    mixed_precision=mp_policy,
    auto_wrap_policy=wrap_policy,
    use_orig_params=True,        # 保留原始参数名，方便 state_dict
    sync_module_states=True,     # rank 0 的权重同步到其他 rank
)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

关键观察：FSDP(model) 这一行就把 ZeRO-3 接好了，
        训练循环和 DDP 几乎一致：loss.backward() + optimizer.step()。


### FSDP2：per-parameter sharding 的新设计

PyTorch 2.4 起引入 FSDP2，是对原 FSDP（现称 FSDP1）的重新设计。FSDP1 以 `nn.Module` 为切分单位，把整个子模块的参数打平成一个 FlatParameter；FSDP2 改为 per-parameter sharding，每个 `nn.Parameter` 独立管理自己的分片。

这个变化解决了几类长期问题。第一，FlatParameter 破坏了 `state_dict` 的结构，保存 checkpoint 时需要额外的 reshape 逻辑；FSDP2 保留每个参数的原始形状，checkpoint 直接对齐预训练权重格式。第二，MoE 场景下专家参数需要独立管理，FlatParameter 把它们打平后很难做 expert parallel；FSDP2 的 per-parameter 粒度天然支持。第三，FSDP1 难以和 DTensor（distributed tensor）组合，FSDP2 直接基于 DTensor 实现，张量并行、流水并行可以正交组合。

API 形状也有变化。FSDP2 不再是包装类，而是函数式调用：

```python
import torch.distributed.tensor.parallel as tp
from torch.distributed.fsdp import fully_shard

model = MyModel()
# 对每个需要分片的子模块调用 fully_shard
for block in model.blocks:
    fully_shard(block)
fully_shard(model)  # 顶层再包一层
```

目前（2025 年中）FSDP2 处于稳定化阶段，PyTorch 主线推荐新项目用 FSDP2，老项目继续用 FSDP1。

## 4. pipeline.sync.Pipe

流水并行（Pipeline Parallelism）把模型按层切成若干 stage，每个 stage 放在不同设备上，数据像流水线一样流过。PyTorch 在 `torch.distributed.pipeline.sync` 提供了 `Pipe` 类做这件事。

构造函数签名：

```python
torch.distributed.pipeline.sync.Pipe(
    module,
    balance,
    devices=None,
    chunks=1,
    checkpoint="except_last",
)

# module: nn.Sequential，必须是顺序结构
# balance: list[int]，每个 stage 包含多少层
# devices: list[device]，每个 stage 放在哪个设备
# chunks: micro-batch 数，越大流水线利用率越高、显存越大
# checkpoint: 'always' / 'except_last' / 'never'，梯度检查点策略
```

`Pipe` 的限制比较硬：模型必须是 `nn.Sequential`，无法表达 skip connection、cross attention 这类跨层连接。现代 LLM 几乎都用残差连接，直接用 `Pipe` 需要把残差改写进每个 stage 内部，工程成本很高。

In [5]:
# === Pipe 伪代码：结构展示（单机无法真实启动多设备流水线） ===
pipe_code = '''
import torch.nn as nn
from torch.distributed.pipeline.sync import Pipe

# 假设 4 层 Transformer block，切成 2 个 stage
model = nn.Sequential(
    Block(0), Block(1),   # stage 0 在 GPU 0
    Block(2), Block(3),   # stage 1 在 GPU 1
)

pipe_model = Pipe(
    model,
    balance=[2, 2],               # stage 0 含 2 层，stage 1 含 2 层
    devices=['cuda:0', 'cuda:1'],
    chunks=8,                     # 8 个 micro-batch 填满流水线
    checkpoint='except_last',     # 除最后一层外都开梯度检查点
)

out = pipe_model(input_batch)
'''

print("PyTorch Pipe 用法：")
print(pipe_code)
print()
print("为什么生产环境很少用 PyTorch 内置 Pipe：")
reasons = [
    "1. 必须是 nn.Sequential，无法表达残差/跨层连接",
    "2. 不支持 1F1B（one-forward-one-backward）调度，bubble 比例高",
    "3. Megatron-LM / DeepSpeed 的 PP 实现做了 interleaved schedule，效率更高",
    "4. Megatron-LM 的 PP 与 TP、EP 联合调优，是工业界事实标准",
]
for r in reasons:
    print(f"  {r}")

PyTorch Pipe 用法：

import torch.nn as nn
from torch.distributed.pipeline.sync import Pipe

# 假设 4 层 Transformer block，切成 2 个 stage
model = nn.Sequential(
    Block(0), Block(1),   # stage 0 在 GPU 0
    Block(2), Block(3),   # stage 1 在 GPU 1
)

pipe_model = Pipe(
    model,
    balance=[2, 2],               # stage 0 含 2 层，stage 1 含 2 层
    devices=['cuda:0', 'cuda:1'],
    chunks=8,                     # 8 个 micro-batch 填满流水线
    checkpoint='except_last',     # 除最后一层外都开梯度检查点
)

out = pipe_model(input_batch)


为什么生产环境很少用 PyTorch 内置 Pipe：
  1. 必须是 nn.Sequential，无法表达残差/跨层连接
  2. 不支持 1F1B（one-forward-one-backward）调度，bubble 比例高
  3. Megatron-LM / DeepSpeed 的 PP 实现做了 interleaved schedule，效率更高
  4. Megatron-LM 的 PP 与 TP、EP 联合调优，是工业界事实标准


PyTorch 内置 `Pipe` 主要用于教学和小规模实验，真实大规模训练几乎都改用 Megatron-LM 的 PipelineParallel 或 DeepSpeed 的 PP 实现。后两者支持 1F1B 调度、interleaved schedule、与张量并行组合，吞吐和显存都优于 `Pipe`。

## 5. Megatron-LM 风格

Megatron-LM 是 NVIDIA 维护的训练代码基线，不是 pip 安装的库。它的核心价值在于把张量并行（TP）和流水并行（PP）封装进了自定义层，用户写的模型代码看起来和普通 PyTorch 几乎一样，但底层自动做切分。

TP 的关键是 `ColumnParallelLinear` 和 `RowParallelLinear` 这两个类。一个标准的 `nn.Linear(in, out)` 在所有 GPU 上持有完整权重；`ColumnParallelLinear` 把权重矩阵按输出维度切分到 N 张卡，每张卡只算自己那一片输出，最后用 all-gather 拼回完整结果。

In [6]:
# === Megatron-LM ColumnParallelLinear 的核心逻辑（伪代码展示切分思想） ===
import torch
import torch.nn as nn
import torch.distributed as dist


class ColumnParallelLinear(nn.Module):
    """按输出维度切分的线性层（教学版，省略通信优化）。

    标准 Linear 权重形状 [out_features, in_features]。
    ColumnParallel 把 out_features 切成 N 份，每张卡只持有 [out/N, in]。
    """

    def __init__(self, in_features, out_features, world_size):
        super().__init__()
        assert out_features % world_size == 0
        self.out_per_partition = out_features // world_size
        self.in_features = in_features
        # 每张卡只存自己那一片权重
        self.weight = nn.Parameter(
            torch.empty(self.out_per_partition, in_features)
        )
        nn.init.normal_(self.weight)

    def forward(self, x):
        # x: [batch, in_features]
        # 每张卡算 [batch, out_per_partition]
        out_local = x @ self.weight.t()
        # 真实 Megatron-LM 用 all-gather 拼回完整输出
        # 这里只展示切分维度
        return out_local


# 演示切分逻辑：8 输出维度切到 4 张卡
world_size = 4
layer = ColumnParallelLinear(in_features=6, out_features=8, world_size=world_size)

print(f"原始 Linear 权重形状: [8, 6]，共 {8*6} 个参数")
print(f"ColumnParallel 权重形状（每卡）: {list(layer.weight.shape)}，")
print(f"  每卡持有 {layer.weight.numel()} 个参数 = 总量 / {world_size}")
print()
print("关键观察：TP 把权重切到 N 卡，")
print("        每卡 forward 只算 [batch, out/N]，")
print("        最后 all-gather 拼回 [batch, out]。")

原始 Linear 权重形状: [8, 6]，共 48 个参数
ColumnParallel 权重形状（每卡）: [2, 6]，
  每卡持有 12 个参数 = 总量 / 4

关键观察：TP 把权重切到 N 卡，
        每卡 forward 只算 [batch, out/N]，
        最后 all-gather 拼回 [batch, out]。


`RowParallelLinear` 是对称设计：按输入维度切分，每张卡持有完整输出维度的一部分输入，forward 时先各自算 `[batch, out]`（完整 out），再 all-reduce 求和。Transformer 的 FFN 通常组合使用：第一层 `ColumnParallelLinear`（升维），第二层 `RowParallelLinear`（降维），中间不需要通信。

### Megatron-LM 源码阅读路线

Megatron-LM 代码量大，初学者容易迷路。建议按这条路线读：

1. **入口**：`pretrain_gpt.py` → 看训练循环骨架，理解 step 的三段式（forward / backward / optimizer step）
2. **模型**：`megatron/models/gpt/gpt_model.py` → 看 GPTModel 如何组装 embedding + transformer blocks + lm_head
3. **并行层**：`megatron/core/tensor_parallel/layers.py` → `ColumnParallelLinear` / `RowParallelLinear` 的实现，重点看 `_initialize_affine_weight` 如何按 rank 切权重
4. **注意力**：`megatron/core/transformer/attention.py` → 看 TP 如何作用在 QKV projection 上（ColumnParallel）和 attention output（RowParallel）
5. **流水并行**：`megatron/core/pipeline_parallel/schedules.py` → 看 1F1B 调度如何交错 forward 和 backward
6. **MoE**：`megatron/core/transformer/moe/` → 专家并行如何与 TP、PP、DP 组合

读完前四步就能理解 TP 的全貌，第五步理解 PP 调度，第六步进入 MoE。

## 6. DeepSpeed ZeRO

DeepSpeed 把 ZeRO 的 Stage 选择、offload 策略、通信优化都做成 JSON 配置项。第 13 节已经展示过 Stage 2 的配置，这里完整展开 `zero_optimization` 的字段结构。

DeepSpeed 的训练循环有三件套：`deepspeed.initialize` 返回 (model_engine, optimizer, dataloader)，后续 forward / backward / step 都通过 model_engine 调用。

In [7]:
# === DeepSpeed ZeRO 配置完整结构 ===
import json

# ZeroStageEnum: 1=切优化器, 2=切梯度+优化器, 3=切参数+梯度+优化器
zero_stage_3_config = {
    "train_micro_batch_size_per_gpu": 4,
    "gradient_accumulation_steps": 8,
    "zero_optimization": {
        "stage": 3,
        "overlap_comm": True,                  # 通信与计算重叠
        "contiguous_gradients": True,          # 梯度连续内存
        "stage3_param_persistence_threshold": 10000,  # 小于这个值的参数不分片
        "offload_optimizer": {
            "device": "none"                   # none / cpu / nvme
        },
        "offload_param": {
            "device": "none"                   # Stage 3 才能开启
        }
    },
    "fp16": {
        "enabled": True,
        "loss_scale": 0,                       # 0 表示动态 loss scaling
        "loss_scale_window": 1000,
        "hysteresis": 2,
        "min_loss_scale": 1
    },
    "activation_checkpointing": {
        "partition_activations": True,         # 激活值也分片
        "cpu_checkpointing": False,
        "number_checkpoints": 16
    }
}

print("DeepSpeed ZeRO Stage 3 配置：")
print(json.dumps(zero_stage_3_config, indent=2, ensure_ascii=False))
print()
print("关键观察：")
print("  - stage=3 等价于 FSDP FULL_SHARD")
print("  - offload_optimizer.device='cpu' = ZeRO-Offload")
print("  - offload_param.device='nvme' = ZeRO-Infinity")
print("  - overlap_comm=True 几乎总是建议开启，提升吞吐")

DeepSpeed ZeRO Stage 3 配置：
{
  "train_micro_batch_size_per_gpu": 4,
  "gradient_accumulation_steps": 8,
  "zero_optimization": {
    "stage": 3,
    "overlap_comm": true,
    "contiguous_gradients": true,
    "stage3_param_persistence_threshold": 10000,
    "offload_optimizer": {
      "device": "none"
    },
    "offload_param": {
      "device": "none"
    }
  },
  "fp16": {
    "enabled": true,
    "loss_scale": 0,
    "loss_scale_window": 1000,
    "hysteresis": 2,
    "min_loss_scale": 1
  },
  "activation_checkpointing": {
    "partition_activations": true,
    "cpu_checkpointing": false,
    "number_checkpoints": 16
  }
}

关键观察：
  - stage=3 等价于 FSDP FULL_SHARD
  - offload_optimizer.device='cpu' = ZeRO-Offload
  - offload_param.device='nvme' = ZeRO-Infinity
  - overlap_comm=True 几乎总是建议开启，提升吞吐


In [8]:
# === DeepSpeed 训练循环三件套（伪代码，需要真实环境运行） ===
deepspeed_loop = '''
import deepspeed

# model / optimizer / dataloader 已构造好
model_engine, optimizer, _, _ = deepspeed.initialize(
    model=model,
    optimizer=optimizer,
    model_parameters=model.parameters(),
    training_data=train_dataset,
    config="deepspeed_config.json",
)

for step, batch in enumerate(dataloader):
    # forward：用 model_engine 而不是 model
    loss = model_engine(batch)

    # backward：用 model_engine.backward 而不是 loss.backward
    model_engine.backward(loss)

    # step：DeepSpeed 内部处理梯度同步 + optimizer.step + zero_grad
    model_engine.step()
'''

print("DeepSpeed 训练循环：")
print(deepspeed_loop)
print("关键差异：")
print("  - loss.backward() → model_engine.backward(loss)")
print("  - optimizer.step() + zero_grad() → model_engine.step()")
print("  - 梯度累积由 config 里的 gradient_accumulation_steps 控制，循环里不写 if")

DeepSpeed 训练循环：

import deepspeed

# model / optimizer / dataloader 已构造好
model_engine, optimizer, _, _ = deepspeed.initialize(
    model=model,
    optimizer=optimizer,
    model_parameters=model.parameters(),
    training_data=train_dataset,
    config="deepspeed_config.json",
)

for step, batch in enumerate(dataloader):
    # forward：用 model_engine 而不是 model
    loss = model_engine(batch)

    # backward：用 model_engine.backward 而不是 loss.backward
    model_engine.backward(loss)

    # step：DeepSpeed 内部处理梯度同步 + optimizer.step + zero_grad
    model_engine.step()

关键差异：
  - loss.backward() → model_engine.backward(loss)
  - optimizer.step() + zero_grad() → model_engine.step()
  - 梯度累积由 config 里的 gradient_accumulation_steps 控制，循环里不写 if


## 7. Accelerate

Accelerate 是 HuggingFace 的薄层封装，把 DDP / FSDP / DeepSpeed 的启动差异藏在一个 `Accelerator` 对象后面。它的设计哲学是：训练循环写得像普通 PyTorch，分布式逻辑由 `Accelerator` 注入。

`Accelerator` 的核心方法：

In [9]:
# === Accelerator 核心方法说明 ===
methods = {
    "accelerator.prepare(*args)": (
        "把 model / optimizer / dataloader / scheduler 包装成分布式版本。"
        "DDP 模式下 model 被包成 DDP，FSDP 模式下被包成 FSDP。"
    ),
    "accelerator.backward(loss)": (
        "取代 loss.backward()。"
        "DeepSpeed 模式下调用 model_engine.backward，"
        "FSDP 模式下处理梯度同步。"
    ),
    "accelerator.accumulate(model)": (
        "上下文管理器，自动处理梯度累积。"
        "配合配置里的 gradient_accumulation_steps。"
    ),
    "accelerator.print(*args)": (
        "只在 rank 0 打印，避免多进程重复输出。"
        "等价于 if accelerator.is_main_process: print(...)。"
    ),
    "accelerator.get_state_dict(model)": (
        "获取 FSDP/DeepSpeed 下的 state_dict，处理分片权重。"
        "checkpoint 保存时必须用这个。"
    ),
    "accelerator.wait_for_everyone()": (
        "barrier 同步。"
        "多进程下保证所有 rank 到达检查点。"
    ),
}

for name, desc in methods.items():
    print(f"{name}:")
    print(f"  {desc}")
    print()

accelerator.prepare(*args):
  把 model / optimizer / dataloader / scheduler 包装成分布式版本。DDP 模式下 model 被包成 DDP，FSDP 模式下被包成 FSDP。

accelerator.backward(loss):
  取代 loss.backward()。DeepSpeed 模式下调用 model_engine.backward，FSDP 模式下处理梯度同步。

accelerator.accumulate(model):
  上下文管理器，自动处理梯度累积。配合配置里的 gradient_accumulation_steps。

accelerator.print(*args):
  只在 rank 0 打印，避免多进程重复输出。等价于 if accelerator.is_main_process: print(...)。

accelerator.get_state_dict(model):
  获取 FSDP/DeepSpeed 下的 state_dict，处理分片权重。checkpoint 保存时必须用这个。

accelerator.wait_for_everyone():
  barrier 同步。多进程下保证所有 rank 到达检查点。



In [10]:
# === 从「朴素训练循环」到「Accelerate 训练循环」的迁移 ===
naive_code = '''
# 朴素 PyTorch 训练循环（单卡）
model = MyModel().cuda()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
for batch in dataloader:
    loss = model(batch)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
'''

accelerate_code = '''
from accelerate import Accelerator

accelerator = Accelerator(
    gradient_accumulation_steps=8,  # 梯度累积写在这里
    mixed_precision="bf16",        # 混合精度写在这里
)

model = MyModel()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# 关键一行：prepare 把对象交给 accelerator 管理
model, optimizer, dataloader = accelerator.prepare(
    model, optimizer, dataloader
)

for batch in dataloader:
    with accelerator.accumulate(model):   # 自动梯度累积
        loss = model(batch)
        accelerator.backward(loss)        # 取代 loss.backward()
        optimizer.step()
        optimizer.zero_grad()
'''

print("迁移前后对比：")
print("=== 朴素循环 ===")
print(naive_code)
print("=== Accelerate 循环 ===")
print(accelerate_code)
print("关键观察：改动只有三处——")
print("  1. 多一个 accelerator 对象")
print("  2. prepare() 包装 model/optimizer/dataloader")
print("  3. loss.backward() → accelerator.backward(loss)")
print("其余训练逻辑完全保留。切换 DDP/FSDP/DeepSpeed 只需改启动配置。")

迁移前后对比：
=== 朴素循环 ===

# 朴素 PyTorch 训练循环（单卡）
model = MyModel().cuda()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
for batch in dataloader:
    loss = model(batch)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

=== Accelerate 循环 ===

from accelerate import Accelerator

accelerator = Accelerator(
    gradient_accumulation_steps=8,  # 梯度累积写在这里
    mixed_precision="bf16",        # 混合精度写在这里
)

model = MyModel()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# 关键一行：prepare 把对象交给 accelerator 管理
model, optimizer, dataloader = accelerator.prepare(
    model, optimizer, dataloader
)

for batch in dataloader:
    with accelerator.accumulate(model):   # 自动梯度累积
        loss = model(batch)
        accelerator.backward(loss)        # 取代 loss.backward()
        optimizer.step()
        optimizer.zero_grad()

关键观察：改动只有三处——
  1. 多一个 accelerator 对象
  2. prepare() 包装 model/optimizer/dataloader
  3. loss.backward() → accelerator.backward(loss)
其余训练逻辑完全保留。切换 D

Accelerate 不是新算法，它的价值是工程层面的：同一份代码，配合 `accelerate launch` 和不同的 config 文件，就能在 DDP、FSDP、DeepSpeed 之间切换。写开源训练脚本、做多后端对比实验时尤其方便。

TRL（Transformer Reinforcement Learning）是建立在 Accelerate 之上的更高层封装，专门做 RLHF / DPO / PPO 训练。它的 `SFTTrainer` / `DPOTrainer` 把数据预处理、训练循环、checkpoint 全部封装好，用户只需要传 model 和 dataset。代价是定制化困难——想改一个训练细节（比如自定义 loss）需要看懂 TRL 的内部结构。

## 8. MoE 场景的框架选择

MoE（Mixture of Experts）训练的并行策略比 dense 模型复杂。除了 DP / TP / PP，还有专家并行（Expert Parallelism，EP）：把不同的专家放到不同 GPU 上，每个 token 通过 all-to-all 路由到对应专家所在的卡。这种拓扑对框架的成熟度要求更高。

主流框架对 MoE 的支持成熟度差异明显：

| 框架 | MoE 支持成熟度 | 典型用途 |
|:---|:---|:---|
| Megatron-LM | 最高（事实标准） | 大规模 MoE 预训练（Mixtral、DeepSeek-V3 等参考实现） |
| DeepSpeed-MoE | 高 | 提供 ZeRO + EP 的组合，PR 协议易迁移 |
| PyTorch FSDP2 | 中（per-parameter 利好 EP） | FSDP1 + EP 较难，FSDP2 改善但仍落后 |
| PyTorch DDP | 低 | 需 `find_unused_parameters=True`，开销大 |
| Accelerate | 取决于后端 | 后端是 Megatron/DeepSpeed 时能力等同后端 |

Megatron-LM 的 `ExpertParallel` 是工业界 MoE 训练的事实标准。它在 `megatron/core/transformer/moe/` 下实现了 expert token 路由、all-to-all 通信、auxiliary loss、load balancing 等完整功能，并且与 TP / PP / DP 的组合调度做了优化。Mixtral、DeepSeek-V3、Qwen-MoE 等开源模型的训练配置都基于 Megatron-LM 或其变体。

In [11]:
# === Megatron-LM MoE 配置的核心字段（结构展示） ===
megatron_moe_config = {
    "num_experts": 64,                    # 全局专家总数
    "expert_model_parallel_size": 8,      # EP 并行度：专家分布到 8 个 TP 组
    "moe_router_topk": 2,                 # 每个 token 路由到 2 个专家
    "moe_aux_loss_coeff": 0.01,           # auxiliary loss 权重，平衡负载
    "moe_z_loss_coeff": 0.001,            # z-loss 防止 router 饱和
    "moe_grouped_gemm": True,             # 用 GroupedGEMM 加速专家计算
    "moe_token_dropping": False,          # 现代 MoE 通常关闭，用 token 重排代替丢弃
}

print("Megatron-LM MoE 核心配置字段：")
for k, v in megatron_moe_config.items():
    print(f"  {k}: {v}")
print()
print("关键观察：")
print("  - expert_model_parallel_size 决定专家如何分布到 GPU")
print("  - moe_router_topk=2 是 Mixtral/DeepSeek-MoE 的典型配置")
print("  - moe_aux_loss_coeff 控制 load balancing 的强度")
print("  - moe_grouped_gemm 是性能关键，避免逐专家 GEMM 的开销")

Megatron-LM MoE 核心配置字段：
  num_experts: 64
  expert_model_parallel_size: 8
  moe_router_topk: 2
  moe_aux_loss_coeff: 0.01
  moe_z_loss_coeff: 0.001
  moe_grouped_gemm: True
  moe_token_dropping: False

关键观察：
  - expert_model_parallel_size 决定专家如何分布到 GPU
  - moe_router_topk=2 是 Mixtral/DeepSeek-MoE 的典型配置
  - moe_aux_loss_coeff 控制 load balancing 的强度
  - moe_grouped_gemm 是性能关键，避免逐专家 GEMM 的开销


DeepSpeed-MoE 是 DeepSpeed 对 MoE 的扩展，提供了 ZeRO + EP 的组合能力。它的优势是配置驱动、与 DeepSpeed 生态兼容；劣势是性能优化（如 GroupedGEMM、shared expert）落后于 Megatron-LM。中小规模 MoE 实验可用 DeepSpeed-MoE 快速跑通，规模上来后通常迁移到 Megatron-LM。

FSDP1 + EP 的组合比较尴尬：FlatParameter 把所有专家的权重打平，无法独立分片。FSDP2 的 per-parameter sharding 改善了这一点，PyTorch 2.5+ 的 `torch.distributed.tensor.parallel` 也加入了 expert parallel 的实验性支持，但截至 2025 年中仍不如 Megatron-LM 成熟。

选型建议：MoE 训练优先考虑 Megatron-LM（或基于它定制的内部框架，如 DeepSeek 的框架、阿袋 PaLM 框架）。研究和原型阶段用 DeepSpeed-MoE 或 FSDP2 + 手写 expert routing 也可行，但要做好性能不如 Megatron-LM 的预期。

## 9. 场景到框架的决策树

把前面几节的选型经验浓缩成一棵决策树。给定一个训练任务，按下面的顺序判断：

```text
1. 模型能完整放进单张 GPU 显存吗？
   是 → DDP（吞吐最高、最简单）
   否 → 进入 2

2. 是 MoE 模型吗？
   是 → Megatron-LM（事实标准，含 EP）
        如果规模较小、想快速验证：DeepSpeed-MoE
   否 → 进入 3

3. 需要张量并行（TP）或流水并行（PP）吗？
   是（比如 100B+ 超大模型）→ Megatron-LM（TP+PP+DP+EP 三维并行）
   否 → 进入 4

4. 只需要数据并行 + ZeRO 省显存？
   偏好 PyTorch 原生、要精细控制 → FSDP（或 FSDP2）
   偏好配置驱动、需要 offload → DeepSpeed
   想一份代码切多后端 → Accelerate + FSDP/DeepSpeed
```

实际项目里经常混用：用 Accelerate 写主脚本，后端选 FSDP 或 DeepSpeed；超大模型直接上 Megatron-LM，不用 Accelerate 这层（Megatron-LM 自己有完整的 launcher 和并行调度）。

## 小结

确认已经理解以下内容：

- [ ] 训练框架 API 分四层：L0 primitive、L1 并行模块、L2 框架层、L3 自建层（Megatron-LM）
- [ ] DDP 的三个关键参数：`device_ids`、`gradient_as_bucket_view`、`find_unused_parameters`
- [ ] FSDP 与 DDP 的区别：参数、梯度、优化器状态全部分片；前向反向时 all-gather 拼回
- [ ] FSDP2 改用 per-parameter sharding，解决 FSDP1 的 FlatParameter 问题，利好 MoE 和 checkpoint
- [ ] PyTorch 内置 `Pipe` 生产环境很少用，多用 Megatron-LM 的 PP（支持 1F1B 调度）
- [ ] Megatron-LM 的 `ColumnParallelLinear` / `RowParallelLinear` 是 TP 的封装核心
- [ ] DeepSpeed 训练循环三件套：`deepspeed.initialize` + `model_engine.backward` + `model_engine.step`
- [ ] Accelerate 的价值是「一份代码切多后端」，底层仍是 DDP/FSDP/DeepSpeed
- [ ] MoE 训练的事实标准是 Megatron-LM，DeepSpeed-MoE 和 FSDP2 是备选
- [ ] 选型决策树：单卡放得下→DDP；MoE→Megatron-LM；超大模型→Megatron-LM；其余→FSDP/DeepSpeed/Accelerate

## 作业

> 可以让 AI 帮忙解释思路、拆步骤、检查方向，但不建议直接让 AI 「做完这道题」。

**作业 1：识别场景对应的框架**

给定以下三个训练场景，写出最合适的框架名称（从 DDP / FSDP / DeepSpeed / Megatron-LM / Accelerate 中选）。

小提示：参考第 9 节的决策树。

In [12]:
# 作业 1：场景到框架的匹配
# 场景 A：在 8 张 A100 上训练 7B dense 模型，单卡放不下完整模型但不需要 TP
# 场景 B：在 2048 张 H100 上训练 100B MoE 模型，需要 TP + PP + EP
# 场景 C：在 2 张消费级 GPU 上微调 1.5B 模型做实验，想一份代码同时支持 DDP 和单卡

answer_a = "FSDP"  # 也可以是 "DeepSpeed"
answer_b = "Megatron-LM"
answer_c = "Accelerate"

valid = {"DDP", "FSDP", "DeepSpeed", "Megatron-LM", "Accelerate"}
assert answer_a in valid, f"场景 A 的答案必须是 {valid} 之一"
assert answer_b in valid, f"场景 B 的答案必须是 {valid} 之一"
assert answer_c in valid, f"场景 C 的答案必须是 {valid} 之一"

# 场景 A：7B 单卡放不下，dense 模型不需要 TP → FSDP 或 DeepSpeed 都对
assert answer_a in {"FSDP", "DeepSpeed"}, (
    "场景 A 提示：7B 单卡放不下需要 ZeRO-3 级切分，dense 不需要 TP"
)
# 场景 B：超大 MoE 需要 TP+PP+EP → Megatron-LM 是事实标准
assert answer_b == "Megatron-LM", (
    "场景 B 提示：100B MoE + TP+PP+EP 只有 Megatron-LM 成熟支持"
)
# 场景 C：想一份代码切多后端 → Accelerate
assert answer_c == "Accelerate", (
    "场景 C 提示：一份代码支持 DDP 和单卡切换，这是 Accelerate 的核心定位"
)

print("作业 1 通过：")
print(f"  场景 A（7B dense, 单卡放不下）: {answer_a}")
print(f"  场景 B（100B MoE, 超大规模）:   {answer_b}")
print(f"  场景 C（小规模实验, 多后端）:   {answer_c}")

作业 1 通过：
  场景 A（7B dense, 单卡放不下）: FSDP
  场景 B（100B MoE, 超大规模）:   Megatron-LM
  场景 C（小规模实验, 多后端）:   Accelerate


**作业 2：补全 FSDP 的 MixedPrecision 配置**

写一个 FSDP 的 MixedPrecision 配置，要求：参数用 BF16，梯度 all-reduce 用 BF16，buffer 也用 BF16。

小提示：参考第 3 节的示例，三个字段名是 `param_dtype`、`reduce_dtype`、`buffer_dtype`。

In [13]:
# 作业 2：FSDP MixedPrecision 配置
import torch
from torch.distributed.fsdp import MixedPrecision

mp_policy = MixedPrecision(
    param_dtype=torch.bfloat16,
    reduce_dtype=torch.bfloat16,
    buffer_dtype=torch.bfloat16,
)

assert mp_policy is not None, "请先构造 MixedPrecision 对象"
assert mp_policy.param_dtype == torch.bfloat16, "param_dtype 应为 torch.bfloat16"
assert mp_policy.reduce_dtype == torch.bfloat16, "reduce_dtype 应为 torch.bfloat16"
assert mp_policy.buffer_dtype == torch.bfloat16, "buffer_dtype 应为 torch.bfloat16"

print("作业 2 通过：")
print(f"  param_dtype:  {mp_policy.param_dtype}")
print(f"  reduce_dtype: {mp_policy.reduce_dtype}")
print(f"  buffer_dtype: {mp_policy.buffer_dtype}")
print("这份配置是 A100/H100 上 FSDP 的标准 BF16 混合精度设置。")

作业 2 通过：
  param_dtype:  torch.bfloat16
  reduce_dtype: torch.bfloat16
  buffer_dtype: torch.bfloat16
这份配置是 A100/H100 上 FSDP 的标准 BF16 混合精度设置。


**作业 3：识别 DDP 在 MoE 场景下需要的参数**

MoE 模型每次 forward 只激活部分专家参数。用 DDP 训练 MoE 时，必须开启哪个参数才能正确同步梯度？

小提示：这个参数会让 DDP 遍历 autograd 图，找出未参与 backward 的参数。参考第 2 节。

In [14]:
# 作业 3：DDP 在 MoE 场景下的关键参数
answer = "find_unused_parameters=True"

valid_params = {
    "find_unused_parameters",
    "find_unused_parameters=True",
    "find_unused_parameters = True",
}
# 标准化：去掉空格和 = True
normalized = answer.replace(" ", "").replace("=True", "").lower() if answer else ""
assert normalized == "find_unused_parameters", (
    "提示：DDP 默认假设所有参数都参与 backward，"
    "MoE 场景下需要开启一个遍历 autograd 图的参数"
)

print("作业 3 通过：")
print(f"  MoE + DDP 必须开启: find_unused_parameters=True")
print("  原因：MoE 每次只激活部分专家，未激活的参数不参与 backward，")
print("        DDP 需要知道这一点才能正确做梯度同步。")
print("  代价：每次 forward 多一次 autograd 图遍历，所以工业界 MoE")
print("        通常直接上 FSDP + expert parallel，不用 DDP。")

作业 3 通过：
  MoE + DDP 必须开启: find_unused_parameters=True
  原因：MoE 每次只激活部分专家，未激活的参数不参与 backward，
        DDP 需要知道这一点才能正确做梯度同步。
  代价：每次 forward 多一次 autograd 图遍历，所以工业界 MoE
        通常直接上 FSDP + expert parallel，不用 DDP。


## 参考资料

- [PyTorch DDP 文档](https://pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html)
- [PyTorch FSDP 文档](https://pytorch.org/docs/stable/fsdp.html)
- [FSDP2 设计文档（PyTorch RFC）](https://github.com/pytorch/pytorch/issues/114299)
- [PyTorch Pipe 文档](https://pytorch.org/docs/stable/pipeline.html)
- [Megatron-LM 仓库](https://github.com/NVIDIA/Megatron-LM)
- [DeepSpeed 文档](https://www.deepspeed.ai/)
- [HuggingFace Accelerate 文档](https://huggingface.co/docs/accelerate/)
- [HuggingFace TRL 文档](https://huggingface.co/docs/trl/)
- Rajbhandari et al., [ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054), 2020
- Shoeybi et al., [Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism](https://arxiv.org/abs/1909.08053), 2019